## Import Library

In [1]:
# 202431077_Siti Shafira Shafarina.A

import csv
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd


## Cek robots.txt (untuk laporan)

In [11]:
# 202431077_Siti Shafira Shafarina.A

from urllib.robotparser import RobotFileParser

URL = "https://en.wikipedia.org/wiki/List_of_tallest_buildings_by_city"
HEADERS = {"User-Agent": "PraktikumScraping/1.0 (Laprak Pemsin; sitishafirashafarina@gmail.com)"}

# Unduh robots.txt dengan requests (bukan rp.read()), lalu serahkan isinya ke parser
r = requests.get("https://en.wikipedia.org/robots.txt", headers=HEADERS, timeout=30)
print("Status robots.txt:", r.status_code)      # harus 200

rp = RobotFileParser()
rp.parse(r.text.splitlines())

print("Boleh diambil:", rp.can_fetch("PraktikumScraping/1.0", URL))   # harus True
print("Pembanding (harus False):",
      rp.can_fetch("PraktikumScraping/1.0", "https://en.wikipedia.org/w/api.php?action=query"))

Status robots.txt: 200
Boleh diambil: True
Pembanding (harus False): False


## Ambil Halaman Web dan Cek status_code

In [3]:
# 202431077_Siti Shafira Shafarina.A

response = requests.get(URL, headers=HEADERS, timeout=30)
print("Status code:", response.status_code)
print("URL akhir  :", response.url)
assert response.status_code == 200, "Gagal mengambil halaman"

Status code: 200
URL akhir  : https://en.wikipedia.org/wiki/List_of_tallest_buildings_by_city


## Ubah Isi Halaman Menjadi Objek BeautifulSoup

In [4]:
# 202431077_Siti Shafira Shafarina.A

soup = BeautifulSoup(response.text, "html.parser")
print("Judul halaman:", soup.title.get_text(strip=True))

Judul halaman: List of tallest buildings by city - Wikipedia


## Cari Semua Blok Data dengan find_all

In [5]:
# 202431077_Siti Shafira Shafarina.A

semua_baris = soup.find_all("tr")
print("Jumlah baris <tr> di halaman:", len(semua_baris))

# Blok data = baris tabel yang punya 9 kolom dan kolom pertama berupa nomor rank
blok_data = []
for baris in semua_baris:
    sel = baris.find_all(["th", "td"])
    if len(sel) == 9 and re.match(r"^\d+", sel[0].get_text(strip=True)):
        blok_data.append(baris)

print("Jumlah blok data:", len(blok_data))
assert len(blok_data) >= 200, "Data kurang dari 200"

Jumlah baris <tr> di halaman: 940
Jumlah blok data: 927


## Ambil Tiap Atribut dari Tiap Blok

In [6]:
# 202431077_Siti Shafira Shafarina.A

def ambil_kolom(baris, nomor):
    """Ambil teks kolom ke-nomor (mulai 0) memakai find dan find_next_sibling."""
    sel = baris.find(["th", "td"])
    for _ in range(nomor):
        sel = sel.find_next_sibling(["th", "td"])
    teks = sel.get_text(strip=True)
    return re.sub(r"\[.*?\]", "", teks).strip()    # hapus catatan kaki seperti [a]

result = []
for baris in blok_data:
    result.append({
        "kota":     ambil_kolom(baris, 1),
        "negara":   ambil_kolom(baris, 2),
        "gedung":   ambil_kolom(baris, 3),
        "tinggi_m": ambil_kolom(baris, 4),
        "lantai":   ambil_kolom(baris, 6),
        "tahun":    ambil_kolom(baris, 7),
    })

print("Jumlah data di list result:", len(result))

Jumlah data di list result: 927


## Tampilkan Hasil dengan Perulangan for

In [7]:
# 202431077_Siti Shafira Shafarina.A

for i, item in enumerate(result[:20], start=1):
    print(i, item)

print("Total data:", len(result))

1 {'kota': 'Dubai', 'negara': 'United Arab Emirates', 'gedung': 'Burj Khalifa', 'tinggi_m': '828', 'lantai': '163', 'tahun': '2010'}
2 {'kota': 'Kuala Lumpur', 'negara': 'Malaysia', 'gedung': 'Merdeka 118', 'tinggi_m': '678.9', 'lantai': '118', 'tahun': '2024'}
3 {'kota': 'Shanghai', 'negara': 'China', 'gedung': 'Shanghai Tower', 'tinggi_m': '632', 'lantai': '128', 'tahun': '2015'}
4 {'kota': 'Mecca', 'negara': 'Saudi Arabia', 'gedung': 'Makkah Royal Clock Tower', 'tinggi_m': '601', 'lantai': '120', 'tahun': '2012'}
5 {'kota': 'Shenzhen', 'negara': 'China', 'gedung': 'Ping An International Finance Centre', 'tinggi_m': '599.1', 'lantai': '115', 'tahun': '2017'}
6 {'kota': 'Tianjin', 'negara': 'China', 'gedung': 'Goldin Finance 117', 'tinggi_m': '596.6', 'lantai': '128', 'tahun': '2027'}
7 {'kota': 'Seoul', 'negara': 'South Korea', 'gedung': 'Lotte World Tower', 'tinggi_m': '554.5', 'lantai': '123', 'tahun': '2017'}
8 {'kota': 'New York City', 'negara': 'United States', 'gedung': 'One Wo

## Simpan ke File CSV

In [8]:
# 202431077_Siti Shafira Shafarina.A

with open("data_gedung.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=result[0].keys())
    writer.writeheader()
    for item in result:
        writer.writerow(item)

print("File data_gedung.csv berhasil dibuat")

File data_gedung.csv berhasil dibuat


## Baca File CSV dengan pandas

In [9]:
# 202431077_Siti Shafira Shafarina.A

df = pd.read_csv("data_gedung.csv")
print(df.head(10))
print()
print(df.dtypes)
print("Jumlah baris:", len(df))

            kota                negara                                gedung  \
0          Dubai  United Arab Emirates                          Burj Khalifa   
1   Kuala Lumpur              Malaysia                           Merdeka 118   
2       Shanghai                 China                        Shanghai Tower   
3          Mecca          Saudi Arabia              Makkah Royal Clock Tower   
4       Shenzhen                 China  Ping An International Finance Centre   
5        Tianjin                 China                    Goldin Finance 117   
6          Seoul           South Korea                     Lotte World Tower   
7  New York City         United States                One World Trade Center   
8      Guangzhou                 China          Guangzhou CTF Finance Centre   
9        Beijing                 China                           CITIC Tower   

   tinggi_m  lantai tahun  
0     828.0     163  2010  
1     678.9     118  2024  
2     632.0     128  2015  
3     6

## Ubah Kolom Angka ke int/float dan Simpan

In [10]:
# 202431077_Siti Shafira Shafarina.A

def ke_angka(x):
    x = str(x).replace(",", "").strip()
    return pd.to_numeric(x, errors="coerce")      

for kolom in ["tinggi_m", "lantai", "tahun"]:
    df[kolom] = df[kolom].apply(ke_angka)

print("Baris yang gagal diubah ke angka:")
print(df[df[["tinggi_m", "lantai", "tahun"]].isna().any(axis=1)])

df = df.dropna(subset=["tinggi_m", "lantai", "tahun"])
df = df.drop_duplicates(subset=["kota", "gedung"])

df["tinggi_m"] = df["tinggi_m"].astype(float)
df["lantai"] = df["lantai"].astype("int64")
df["tahun"] = df["tahun"].astype("int64")

print()
print(df.dtypes)
print("Jumlah data akhir:", len(df))
assert len(df) >= 200, "Data akhir kurang dari 200"

df.to_csv("data_gedung_bersih.csv", index=False)
print("File data_gedung_bersih.csv berhasil disimpan")
df.head(10)

Baris yang gagal diubah ke angka:
           kota       negara          gedung  tinggi_m  lantai  tahun
50    Pyongyang  North Korea  Ryugyong Hotel     330.0     105    NaN
332       Cairo        Egypt     Fouda Tower     166.0      50    NaN
394  Alexandria        Egypt  El Salam Tower     155.0      36    NaN

kota         object
negara       object
gedung       object
tinggi_m    float64
lantai        int64
tahun         int64
dtype: object
Jumlah data akhir: 923
File data_gedung_bersih.csv berhasil disimpan


,kota,negara,gedung,tinggi_m,lantai,tahun
0,Dubai,United Arab Emirates,Burj Khalifa,828.0,163,2010
1,Kuala Lumpur,Malaysia,Merdeka 118,678.9,118,2024
2,Shanghai,China,Shanghai Tower,632.0,128,2015
3,Mecca,Saudi Arabia,Makkah Royal Clock Tower,601.0,120,2012
4,Shenzhen,China,Ping An International Finance Centre,599.1,115,2017
5,Tianjin,China,Goldin Finance 117,596.6,128,2027
6,Seoul,South Korea,Lotte World Tower,554.5,123,2017
7,New York City,United States,One World Trade Center,541.3,104,2014
8,Guangzhou,China,Guangzhou CTF Finance Centre,530.0,111,2016
9,Beijing,China,CITIC Tower,527.7,109,2018
